# 07 — Game-statistics deep dive (team style profiles)

Turns parsed replays into a **style vector** per team: economy timing
curves, laning strength by role, objective control, teamfight economics,
pace, and lead-conversion. These are the features that explain *how* a
team wins — and they feed both the matchup model (03) and the
derivative-market models (08).

Requires parsed replays: rows where `parsed == True`. Unparsed matches
silently carry null timelines, so always check coverage first.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
pd.set_option('display.max_columns', 60); pd.set_option('display.width', 160)
DATA = ROOT / 'data'


In [ ]:
from src.game_stats import (team_style_profile, timing_features, lane_features,
                            objective_features, teamfight_features,
                            pace_features, to_team_long, opponent_adjust)
matches = pd.read_parquet(DATA / 'matches.parquet')
mp      = pd.read_parquet(DATA / 'match_players.parquet')
teams   = pd.read_parquet(DATA / 'teams.parquet')[['team_id','name']]
cov = matches.parsed.mean() if 'parsed' in matches.columns else float('nan')
print(f'parsed-replay coverage: {cov:.0%}  (timeline/objective features need this)')
prof = team_style_profile(matches, mp, teams)
prof.head(16)

In [ ]:
# Timing profile: who is ahead when? scaling_slope>0 = grows leads late
tf = timing_features(matches).merge(teams, on='team_id', how='left')
cols = [c for c in tf.columns if c.startswith('gold_adv')] + ['scaling_slope']
tf.nlargest(16, 'gold_adv_20')[['name'] + cols].round(0)

In [ ]:
import matplotlib.pyplot as plt
top = prof.nlargest(8, 'games')
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
mins = [10, 15, 20, 25, 30, 40]
for _, r in top.iterrows():
    ys = [r.get(f'gold_adv_{m}') for m in mins]
    ax[0].plot(mins, ys, marker='o', label=str(r.get('name', r.team_id))[:16])
ax[0].axhline(0, color='k', lw=.8); ax[0].set(xlabel='minute', ylabel='mean gold adv',
    title='Economy timing curves')
ax[0].legend(fontsize=7)
if {'close_rate','comeback_rate'} <= set(prof.columns):
    ax[1].scatter(prof.close_rate, prof.comeback_rate, alpha=.6)
    for _, r in top.iterrows():
        ax[1].annotate(str(r.get('name'))[:12], (r.close_rate, r.comeback_rate), fontsize=7)
    ax[1].set(xlabel='closes leads (win | +3k at 20)', ylabel='comebacks (win | -3k at 20)',
              title='Lead conversion')
plt.tight_layout()

In [ ]:
# Objectives & teamfights: control vs chaos
obj = objective_features(matches).merge(teams, on='team_id', how='left')
tfi = teamfight_features(matches).merge(teams, on='team_id', how='left')
display(obj.nlargest(12, 'first_tower_rate').round(3))
display(tfi.nlargest(12, 'fight_gold_swing').round(1))

In [ ]:
# OPPONENT ADJUSTMENT — the anti-soft-schedule guard.
from src.model import fit_elo
elo, _ = fit_elo(matches)
long = to_team_long(matches)
long['gold20'] = long.get('gold_adv_20') * long['sign'] if 'gold_adv_20' in long else np.nan
adj = opponent_adjust(long, 'gold20', elo)
adj.merge(teams, on='team_id', how='left').nlargest(15, 'gold20_adj').round(0)

In [ ]:
# RADIANT/DIRE split per team + tournament-wide rate (feeds notebook 08)
pf = pace_features(matches)
long = to_team_long(matches)
rad_rate = long.loc[long.is_radiant, 'win'].mean()
print(f'sample-wide RADIANT win rate: {rad_rate:.4f}  (n={long.is_radiant.sum():,} games)')
print('use this as prior_p in derivatives.radiant_dire_market -- but filter to the CURRENT PATCH first:')
cur = matches.patch.max()
lc = to_team_long(matches[matches.patch == cur])
print(f'current patch ({cur}) radiant win rate: {lc.loc[lc.is_radiant, "win"].mean():.4f} (n={lc.is_radiant.sum():,})')